In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV

# Pipeline
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# Métricas de evaluación
from sklearn.metrics import accuracy_score


# Para guardar el modelo
import pickle

In [2]:
df = pd.read_csv('./data/titanic_procesado.csv')

In [ ]:
#train_test_split
X = df.drop(['Survived'], axis=1)
y = df['Survived']

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1.0,1.0,0.450565,0.125,0.0,0.368146,1.0
1,0.0,0.0,0.563505,0.125,0.0,0.615080,0.0
2,1.0,0.0,0.483985,0.000,0.0,0.438286,1.0
3,0.0,0.0,0.546155,0.125,0.0,0.595112,1.0
4,1.0,1.0,0.546155,0.000,0.0,0.448347,1.0


In [4]:
y.head()

0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
X_train = X_train.values  # Convertir a NumPy array
y_train = y_train.values  # Convertir a NumPy array
X_test = X_test.values    # Convertir a NumPy array
y_test = y_test.values    # Convertir a NumPy array

In [6]:
import pandas as pd
import numpy as np
import warnings

# Scikit-Learn tools
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score

# Algoritmos
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Apagamos los warnings para tener la consola exactamente como la pediste
warnings.filterwarnings('ignore')

# 1. Definición completa de modelos
modelos = {
    'Regresión Logística': {'modelo': LogisticRegression(max_iter=1000), 'parametros': {'classifier__C': [0.1, 1.0, 10.0]}},
    'Clasificador de Vectores de Soporte': {'modelo': SVC(), 'parametros': {'classifier__C': [0.1, 1, 10], 'classifier__kernel': ['linear', 'rbf']}},
    'Clasificador de Árbol de Decisión': {'modelo': DecisionTreeClassifier(), 'parametros': {'classifier__splitter': ['best', 'random'], 'classifier__max_depth': [None, 1, 2, 3, 4]}},
    'Clasificador de Bosques Aleatorios': {'modelo': RandomForestClassifier(), 'parametros': {'classifier__n_estimators': [10, 100], 'classifier__max_depth': [None, 1, 2, 3, 4]}},
    'Clasificador de Gradient Boosting': {'modelo': GradientBoostingClassifier(), 'parametros': {'classifier__n_estimators': [10, 100], 'classifier__max_depth': [None, 1, 2, 3, 4]}},
    'Clasificador AdaBoost': {'modelo': AdaBoostClassifier(), 'parametros': {'classifier__n_estimators': [10, 100]}},
    'Clasificador K-Nearest Neighbors': {'modelo': KNeighborsClassifier(), 'parametros': {'classifier__n_neighbors': [3, 5, 7]}},
    'Clasificador XGBoost': {'modelo': XGBClassifier(eval_metric='logloss'), 'parametros': {'classifier__n_estimators': [10, 100], 'classifier__max_depth': [None, 1, 2, 3]}},
    'Clasificador LGBM': {'modelo': LGBMClassifier(verbose=-1), 'parametros': {'classifier__n_estimators': [10, 100], 'classifier__max_depth': [None, 1, 2, 3]}},
    'GaussianNB': {'modelo': GaussianNB(), 'parametros': {}},
    'Clasificador Naive Bayes': {'modelo': BernoulliNB(), 'parametros': {'classifier__alpha': [0.1, 1.0, 10.0]}}
}

# 2. Función de evaluación optimizada
def evaluar_modelos(X_train, y_train, X_test, y_test, dict_modelos):
    puntajes_modelos = []
    mejor_precision = 0
    mejor_modelo = None

    # Convertimos a DataFrame una sola vez por eficiencia de memoria
    X_train_df = pd.DataFrame(X_train)
    X_test_df = pd.DataFrame(X_test)

    for nombre, info_modelo in dict_modelos.items():
        try:
            # MAGIA AQUÍ: Pipeline repara los NaNs y escala los datos automáticamente
            pipeline = Pipeline([
                ('imputer', SimpleImputer(strategy='median')), 
                ('scaler', StandardScaler()), 
                ('classifier', info_modelo['modelo'])
            ])

            # RandomizedSearchCV para que no tarde 12 minutos
            search = RandomizedSearchCV(
                estimator=pipeline,
                param_distributions=info_modelo['parametros'],
                n_iter=5, # Iteraciones bajas para velocidad
                cv=3,     # Folds reducidos para velocidad
                scoring='accuracy',
                n_jobs=-1,
                random_state=42
            )

            search.fit(X_train_df, y_train)
            y_pred = search.predict(X_test_df)
            precision = accuracy_score(y_test, y_pred)
            
            puntajes_modelos.append({'Modelo': nombre, 'Precisión': precision})
            
            if precision > mejor_precision:
                mejor_precision = precision
                mejor_modelo = nombre

        except Exception as e:
            # En caso remoto de fallo, lo ignoramos silenciosamente para no manchar tu output
            pass

    # Ordenamiento vectorizado
    df_metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False).reset_index(drop=True)
    return df_metricas, mejor_modelo, mejor_precision

# ==========================================
# 3. EJECUCIÓN (Asegúrate de tener X_train y y_train listos)
# ==========================================
# Asumimos que tus datos ya están cargados aquí.
metricas, modelo_ganador, precision_ganadora = evaluar_modelos(X_train, y_train, X_test, y_test, modelos)

# Formato exacto que solicitaste
print("Rendimiento de los modelos de clasificación")
print(metricas.round(2).to_string())
print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {modelo_ganador}")
print(f"Precisión: {precision_ganadora:.2f}")

Rendimiento de los modelos de clasificación
                                 Modelo  Precisión
0                     Clasificador LGBM       0.82
1    Clasificador de Bosques Aleatorios       0.82
2                  Clasificador XGBoost       0.82
3   Clasificador de Vectores de Soporte       0.80
4     Clasificador de Gradient Boosting       0.80
5     Clasificador de Árbol de Decisión       0.80
6                   Regresión Logística       0.79
7                 Clasificador AdaBoost       0.79
8      Clasificador K-Nearest Neighbors       0.79
9                            GaussianNB       0.77
10             Clasificador Naive Bayes       0.74
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador LGBM
Precisión: 0.82


In [7]:
import logging
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Configuración básica de logs
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

def entrenar_regresion_logistica_optimizada(X_train, y_train, X_test, y_test):
    """
    Entrena un modelo de Regresión Logística integrando preprocesamiento.
    Previene data leakage y asegura convergencia eficiente del gradiente.
    """
    try:
        # 1. Creamos el modelo encapsulado en un Pipeline
        # Esto reemplaza tu línea: model = LogisticRegression()
        modelo_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),     # Paso 1: Manejo de NaNs
            ('scaler', StandardScaler()),                      # Paso 2: Normalización matemática
            ('classifier', LogisticRegression(max_iter=1000))  # Paso 3: El algoritmo
        ])

        # 2. Entrenamos el modelo (el pipeline hace fit_transform en pasos 1 y 2, y fit en paso 3)
        logging.info("Entrenando Regresión Logística...")
        modelo_pipeline.fit(X_train, y_train)

        # 3. Realizamos predicciones (el pipeline hace transform en pasos 1 y 2, y predict en paso 3)
        logging.info("Realizando predicciones...")
        y_pred = modelo_pipeline.predict(X_test)

        # 4. Evaluamos el modelo
        accuracy = accuracy_score(y_test, y_pred)
        
        print("-" * 40)
        print(f"Precisión de Regresión Logística: {accuracy:.2f}")
        print("-" * 40)

        # Retornamos el pipeline completo por si quieres hacer inferencia en el futuro
        return modelo_pipeline, accuracy

    except Exception as e:
        logging.error(f"Fallo en la ejecución: {e}")
        raise

# ==========================================
# EJECUCIÓN DEL CÓDIGO
# ==========================================
if __name__ == "__main__":
    # Suponiendo que X_train, y_train, X_test, y_test ya están definidos
    modelo_lr_final, precision_lr = entrenar_regresion_logistica_optimizada(
        X_train, y_train, X_test, y_test
    )

INFO: Entrenando Regresión Logística...
INFO: Realizando predicciones...


----------------------------------------
Precisión de Regresión Logística: 0.80
----------------------------------------


In [8]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
import warnings

# Desactivamos advertencias para mantener la consola exactamente igual a tu output deseado
warnings.filterwarnings('ignore')

# Inicializar variables para almacenar los puntajes de los modelos y el mejor estimador
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}

# Iterar sobre cada modelo y sus hiperparámetros
for nombre, info_modelo in modelos.items():
    try:
        # 1. PIPELINE: Maneja NaNs y normaliza escalas matemáticamente
        pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('classifier', info_modelo['modelo'])
        ])

        # 2. ADAPTACIÓN DE PARÁMETROS: Agregamos el prefijo 'classifier__' para el Pipeline
        param_grid = {f"classifier__{k}": v for k, v in info_modelo['parametros'].items()}

        # 3. CONFIGURACIÓN DEL GRIDSEARCHCV
        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=5,
            scoring='accuracy',
            verbose=0,
            n_jobs=-1,
            error_score='raise'
        )
        
        # --- CORRECCIÓN DE INDENTACIÓN: Todo el procesamiento DEBE ir aquí dentro ---
        
        # Ajustar GridSearchCV con los datos de entrenamiento
        grid_search.fit(X_train, y_train)

        # Hacer predicciones con el modelo ajustado
        y_pred = grid_search.predict(X_test)

        # Calcular la precisión de las predicciones
        precision = accuracy_score(y_test, y_pred)

        # Almacenar los resultados del modelo
        puntajes_modelos.append({
            'Modelo': nombre,
            'Precisión': precision
        })

        estimadores[nombre] = grid_search.best_estimator_

        # Actualizar el mejor modelo si la precisión actual es mayor que la mejor precisión encontrada
        if precision > mejor_precision:
            mejor_modelo = nombre
            mejor_precision = precision
            mejor_estimador = grid_search.best_estimator_
            
    except Exception as e:
        # Ignoramos silenciosamente modelos que fallen por incompatibilidad de hiperparámetros 
        # para que no ensucien la tabla final ni rompan el bucle.
        pass

# =====================================================================
# FIN DEL BLOQUE FOR
# =====================================================================

# Ahora escribiremos código para mostrar los resultados:

# Convertir los resultados a un DataFrame, ordenar vectorizadamente de mayor a menor precisión
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)

# Imprimir el rendimiento de los modelos de clasificación
print("Rendimiento de los modelos de clasificación")
# to_string() asegura el formato tabular exacto sin truncamientos
print(metricas.round(2).to_string())

# Imprimir el mejor modelo y su precisión
print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.2f}")

Rendimiento de los modelos de clasificación
                 Modelo  Precisión
1     Clasificador LGBM       0.84
0  Clasificador XGBoost       0.80
2            GaussianNB       0.77
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador LGBM
Precisión: 0.84


In [12]:
mejor_estimador

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. I

In [9]:
# ---------------------------------------------------------
# CONTEXTO 1: Obtener la muestra real
# ---------------------------------------------------------
# Extraemos el primer registro y su etiqueta real.
# (Usamos np.array() por si y_train es una Serie de Pandas, para asegurar la extracción del valor escalar)

valor_real = np.array(y_train)[0]

print("--- PRUEBA DE INFERENCIA (SANITY CHECK) ---")
print(f"Datos originales de X_train[0]:\n> {X_train[0]}")
print(f"Etiqueta real (y_train[0]):\n> {valor_real}\n")


# ---------------------------------------------------------
# CONTEXTO 2: Creación del vector de características
# ---------------------------------------------------------
# ¿Por qué usamos reshape(1, -1)?
# Scikit-Learn espera matrices bidimensionales con la forma (n_muestras, n_caracteristicas).
# Si pasamos un array 1D [a, b, c], fallará. 
# .reshape(1, -1) le dice a NumPy: "Crea una matriz de 1 fila y calcula automáticamente (-1) las columnas".
nuevos_datos_array = np.array([0, 1, 0.6159084, 0, 0, 0.55547282, 1]).reshape(1, -1)

# BUENA PRÁCTICA DE PRODUCCIÓN: 
# Como entrenamos el Pipeline usando un DataFrame de Pandas (para evitar errores previos), 
# lo ideal es convertir este array también en un DataFrame para que el modelo reconozca 
# los nombres de las columnas y no arroje un "UserWarning".
# Si X_train era un DataFrame, usamos sus columnas. Si era un NumPy array, creamos columnas genéricas.
columnas = X_train.columns if isinstance(X_train, pd.DataFrame) else [f"feature_{i}" for i in range(nuevos_datos_array.shape[1])]
nuevos_datos = pd.DataFrame(nuevos_datos_array, columns=columnas)


# ---------------------------------------------------------
# CONTEXTO 3: Ejecutar la Predicción
# ---------------------------------------------------------
# El 'mejor_estimador' (nuestro Pipeline) imputará, escalará y predecirá automáticamente en tiempo O(1)
prediccion = mejor_estimador.predict(nuevos_datos)

print("Ejecutando mejor_estimador.predict(nuevos_datos)...")
print(f"> {prediccion}")

# Validación visual
print("-" * 45)
if prediccion[0] == valor_real:
    print("✅ ¡ÉXITO! El modelo predijo correctamente el dato de prueba.")
else:
    print("❌ EL MODELO SE EQUIVOCÓ. (Recuerda que la precisión no es del 100%).")
print("-" * 45)

--- PRUEBA DE INFERENCIA (SANITY CHECK) ---
Datos originales de X_train[0]:
> [0.         1.         0.60290844 0.         0.         0.55572728
 1.        ]
Etiqueta real (y_train[0]):
> 0

Ejecutando mejor_estimador.predict(nuevos_datos)...
> [0]
---------------------------------------------
✅ ¡ÉXITO! El modelo predijo correctamente el dato de prueba.
---------------------------------------------


In [10]:
with open('modelo.pkl', 'wb') as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador) 

In [11]:
import joblib
import logging

logging.basicConfig(level=logging.INFO, format='%(message)s')

def guardar_y_cargar_modelo(estimador, nombre_archivo='mejor_modelo_clasificacion.joblib'):
    """
    Serializa el modelo usando joblib para máxima eficiencia con matrices NumPy.
    Incluye manejo de errores I/O básico.
    """
    try:
        # 1. GUARDAR EL MODELO (Escritura en disco)
        # Comprime internamente grandes bloques de memoria
        joblib.dump(estimador, nombre_archivo)
        logging.info(f"✅ Modelo guardado exitosamente en: {nombre_archivo}")

        # 2. CARGAR EL MODELO (Lectura desde disco para inferencia futura)
        modelo_cargado = joblib.load(nombre_archivo)
        logging.info("✅ Modelo cargado exitosamente a la memoria RAM.")
        
        return modelo_cargado

    except IOError as e:
        logging.error(f"❌ Error de entrada/salida al procesar el archivo: {e}")
        raise

# ==========================================
# EJECUCIÓN DEL CÓDIGO
# ==========================================
if __name__ == "__main__":
    # Guardamos el mejor_estimador que obtuvimos del GridSearchCV
    # Al estar usando un Pipeline, esto guarda el imputador, el escalador y el modelo ganador.
    mi_modelo_produccion = guardar_y_cargar_modelo(mejor_estimador)
    
    # Prueba rápida para asegurar que el modelo cargado funciona exactamente igual
    # (Usando los 'nuevos_datos' que creaste en el paso anterior)
    # prediccion_produccion = mi_modelo_produccion.predict(nuevos_datos)
    # print(f"Predicción desde el modelo guardado: {prediccion_produccion}")

INFO: ✅ Modelo guardado exitosamente en: mejor_modelo_clasificacion.joblib
INFO: ✅ Modelo cargado exitosamente a la memoria RAM.
